# 🚗 Car Dataset — Real-time Pipeline
Voer de cellen van boven naar beneden uit.

## 1 · Imports & Configuratie

In [10]:
import pandas as pd
import logging
import os
import io
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient
import glob


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger(__name__)


INPUT_FOLDER = './dataset'
OUTPUT_PATH  = './output/cars_processed.csv'
BLOB_NAME    = 'cars/cars_processed.csv'

## 2 · Reader

In [16]:
files = glob.glob(os.path.join(INPUT_FOLDER, '*.csv'))

if not files:
    logger.error(f"Geen CSV gevonden in {INPUT_FOLDER}")
else:
    file_path = files[0]
    try:
        df = pd.read_csv(file_path)
        logger.info(f"Gelezen: {file_path} — {df.shape[0]:,} rijen x {df.shape[1]} kolommen")
    except Exception as e:
        logger.error(f"Fout bij inlezen {file_path}: {e}")

2026-05-03 19:12:07,656 - INFO - Gelezen: ./dataset/cars.csv — 160 rijen x 13 kolommen


## 3 · Validator
Elke check logt het **aantal** slechte rijen, de **rij-indices** én de **waarden** die afgekeurd werden.

In [ ]:
os.makedirs('./logs', exist_ok=True)
log_path = './logs/cars_validation.log'


def log(msg):
    with open(log_path, 'a', encoding='UTF-8') as f:
        f.write(msg + '\n')


log('=' * 60)
log(f"Validation run — {pd.Timestamp.now()}")
log(f"Rows before: {len(df)}, Columns: {len(df.columns)}")
log('=' * 60)


# ── Duplicates ────────────────────────────────────────────────────────────
dup_mask = df.duplicated()
if dup_mask.any():
    dup_rows = df[dup_mask]
    log(f"[DROP] duplicates: {dup_mask.sum()} dubbele rijen")
    log(f"       Indices: {dup_rows.index.tolist()}")
    log(f"       car_ids: {dup_rows['car_id'].tolist()}")
    df = df[~dup_mask].reset_index(drop=True)


# ── Mandatory null checks ─────────────────────────────────────────────────
for col in ['car_id', 'brand', 'year', 'mileage_km', 'fuel_type', 'price_eur']:
    mask = df[col].isna()
    if mask.any():
        bad = df[mask]
        log(f"[DROP] {col}: {mask.sum()} null values")
        log(f"       Indices: {bad.index.tolist()}")
        log(f"       car_ids: {bad['car_id'].tolist()}")
        df = df[~mask].reset_index(drop=True)


# ── brand: valid brands ───────────────────────────────────────────────────
valid_brands = ['Toyota','Ford','BMW','Mercedes','Volkswagen','Audi','Honda','Hyundai',
                'Renault','Peugeot','Skoda','Seat','Nissan','Kia','Mazda','Volvo',
                'Citroën','Dacia','Opel','Fiat']
mask = ~df['brand'].isin(valid_brands)
if mask.any():
    bad = df[mask]
    log(f"[DROP] brand: {mask.sum()} invalid values")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       Values: {bad['brand'].tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── year: 1990–2025 ───────────────────────────────────────────────────────
mask = (df['year'] < 1990) | (df['year'] > 2025)
if mask.any():
    bad = df[mask]
    log(f"[DROP] year: {mask.sum()} out of range (1990-2025)")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       Values: {bad['year'].tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── mileage_km: >= 0 ──────────────────────────────────────────────────────
mask = df['mileage_km'] < 0
if mask.any():
    bad = df[mask]
    log(f"[DROP] mileage_km: {mask.sum()} negative values")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       Values: {bad['mileage_km'].tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── fuel_type: valid values ───────────────────────────────────────────────
valid_fuels = ['Petrol', 'Diesel', 'Electric', 'Hybrid']
mask = ~df['fuel_type'].isin(valid_fuels)
if mask.any():
    bad = df[mask]
    log(f"[DROP] fuel_type: {mask.sum()} invalid values")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       Values: {bad['fuel_type'].tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── price_eur: > 0 ────────────────────────────────────────────────────────
mask = df['price_eur'] <= 0
if mask.any():
    bad = df[mask]
    log(f"[DROP] price_eur: {mask.sum()} non-positive values")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       Values: {bad['price_eur'].tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── engine_cc: null → DROP, <= 0 → DROP ──────────────────────────────────
mask = df['engine_cc'].isna() | (df['engine_cc'] <= 0)
if mask.any():
    bad = df[mask]
    log(f"[DROP] engine_cc: {mask.sum()} null or non-positive values")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── horsepower: null → DROP, <= 0 → DROP ─────────────────────────────────
mask = df['horsepower'].isna() | (df['horsepower'] <= 0)
if mask.any():
    bad = df[mask]
    log(f"[DROP] horsepower: {mask.sum()} null or non-positive values")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── num_doors: null → DROP, invalid → DROP ───────────────────────────────
mask = df['num_doors'].isna() | ~df['num_doors'].isin([2, 3, 4, 5])
if mask.any():
    bad = df[mask]
    log(f"[DROP] num_doors: {mask.sum()} null or invalid values (expected 2-5)")
    log(f"       Indices: {bad.index.tolist()}")
    log(f"       car_ids: {bad['car_id'].tolist()}")
    df = df[~mask].reset_index(drop=True)


# ── co2_g_per_km: Electric null → 0, others null → DROP, < 0 → DROP ──────
mask_null = df['co2_g_per_km'].isna()
if mask_null.any():
    electric_null = mask_null & (df['fuel_type'] == 'Electric')
    other_null = mask_null & (df['fuel_type'] != 'Electric')
    if electric_null.any():
        log(f"[FLAG] co2_g_per_km: {electric_null.sum()} Electric null values — imputed 0")
        log(f"       Indices: {df[electric_null].index.tolist()}")
        log(f"       car_ids: {df[electric_null]['car_id'].tolist()}")
        df.loc[electric_null, 'co2_g_per_km'] = 0
    if other_null.any():
        bad = df[other_null]
        log(f"[DROP] co2_g_per_km: {other_null.sum()} null values (non-Electric) — DROPPED")
        log(f"       Indices: {bad.index.tolist()}")
        log(f"       car_ids: {bad['car_id'].tolist()}")
        df = df[~other_null].reset_index(drop=True)

mask = df['co2_g_per_km'] < 0
if mask.any():
    bad = df[mask]
    log(f"[DROP] co2_g_per_km: {mask.sum()} negative values")
    log(f"       Indices: {bad.index.tolist()}, Values: {bad['co2_g_per_km'].tolist()}")
    df = df[~mask].reset_index(drop=True)

log(f"Rows after: {len(df)}")
log("Validation complete.\n")
print(f"Done. Rows remaining: {len(df)}. Check logs/cars_validation.log")

Gedaan. Rijen over: 9. Check logs/cars_validation.log


## 4 · Processor

In [13]:
current_year = pd.Timestamp.now().year

df['car_age_years'] = current_year - df['year'].astype(int)

df['price_category'] = pd.cut(
    df['price_eur'],
    bins=[0, 10000, 30000, 60000, float('inf')],
    labels=['Budget', 'Mid-range', 'Premium', 'Luxury'],
    right=False
)

df['mileage_category'] = pd.cut(
    df['mileage_km'],
    bins=[0, 50000, 150000, float('inf')],
    labels=['Low', 'Medium', 'High'],
    right=False,
    include_lowest=True
)

df['hp_per_100cc'] = (df['horsepower'] / df['engine_cc'] * 100).round(2)

df['is_electric'] = (df['fuel_type'] == 'Electric').astype(int)

logger.info(f"Processing compleet: {df.shape[0]:,} rijen x {df.shape[1]} kolommen.")
df.head()


2026-05-03 16:33:10,057 - INFO - Processing compleet: 9 rijen x 18 kolommen.


,car_id,brand,model,year,mileage_km,fuel_type,engine_cc,horsepower,price_eur,color,transmission,num_doors,co2_g_per_km,car_age_years,price_category,mileage_category,hp_per_100cc,is_electric
0,1,Toyota,New_Model_1,2014,113500,Petrol,3115,87,41233,White,Automatic,3,235,12,Premium,Medium,2.79,0
1,2,Renault,New_Model_2,2020,43984,Diesel,3851,182,27987,Black,Manual,2,63,6,Mid-range,Low,4.73,0
2,3,Peugeot,New_Model_3,2011,47018,Hybrid,3599,238,37542,Silver,Automatic,5,204,15,Premium,Low,6.61,0
3,5,Volvo,New_Model_5,2023,206010,Petrol,1560,252,38807,Red,Manual,5,120,3,Premium,High,16.15,0
4,6,Nissan,New_Model_6,2014,131181,Diesel,4877,205,37684,Blue,Automatic,2,8,12,Premium,Medium,4.20,0


## 5 · Writer

In [14]:
load_dotenv()

OUTPUT_PATH = 'output/cars.csv'

os.makedirs('output', exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

client = BlobServiceClient.from_connection_string(os.getenv('AZURE_STORAGE_CONNECTION_STRING'))
blob   = client.get_blob_client(container="yellow-taxi-data", blob="cars.csv")

with open(OUTPUT_PATH, 'rb') as f:
    blob.upload_blob(f, overwrite=True)

print("Klaar. Opgeslagen lokaal en naar Azure.")

2026-05-03 16:33:10,080 - INFO - Request URL: 'https://yellowtaxinachatkaran.blob.core.windows.net/yellow-taxi-data/cars.csv'
Request method: 'PUT'
Request headers:
    'Content-Length': '1071'
    'x-ms-blob-type': 'REDACTED'
    'x-ms-version': 'REDACTED'
    'Content-Type': 'application/octet-stream'
    'Accept': 'application/xml'
    'User-Agent': 'azsdk-python-storage-blob/12.28.0 Python/3.13.7 (macOS-26.3.1-arm64-arm-64bit-Mach-O)'
    'x-ms-date': 'REDACTED'
    'x-ms-client-request-id': '02150958-46fd-11f1-a0b4-f6136239a883'
    'Authorization': 'REDACTED'
A body is sent with the request
2026-05-03 16:33:10,284 - INFO - Response status: 201
Response headers:
    'Content-Length': '0'
    'Content-MD5': 'REDACTED'
    'Last-Modified': 'Sun, 03 May 2026 14:33:10 GMT'
    'ETag': '"0x8DEA920E6673995"'
    'Server': 'Windows-Azure-Blob/1.0 Microsoft-HTTPAPI/2.0'
    'x-ms-request-id': 'e40d23a5-b01e-005c-4b09-db76c7000000'
    'x-ms-client-request-id': '02150958-46fd-11f1-a0b4-f61

Klaar. Opgeslagen lokaal en naar Azure.
